# BKQuiz: Base · CPT · SFT · CPT+SFT
Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**.

Notebook tạo nhánh **SFT-only từ base** (281 train,40 validation, cùng1epoch/cấu hình)
vì ZIP bạn gửi là **CPT+SFT**. Không train lại CPT hoặc CPT+SFT. Mỗi adapter nạp riêng lên cùng base Qwen3-1.7B pinned revision.

Đánh giá cả4 trên cùng40 test: assistant NLL/PPL; chọn đáp án đóng sách; sinh quiz JSON theo ngữ cảnh.
Mặc định sinh quiz trên12 câu phủ các học phần, có thể đổi thành40 trước lần chạy đầu.
Test không dùng chọn checkpoint. Một seed và40 test chỉ là đối chứng nhỏ, có tài liệu cùng nguồn với train.
Đáp án nguồn chưa phải expert gold; JSON hợp lệ chưa chứng minh lời giải sâu. Không dùng teacher/VLM.

Kết quả lưu Drive, cache từng câu để tiếp tục. Không có cam kết hoàn thành dưới1h; thời gian sinh2048tokens/câu phụ thuộc GPU và đầu ra.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

In [ ]:
import os, sys, json, hashlib, shutil, zipfile, stat
from pathlib import PurePosixPath
DRIVE_ROOT=Path('/content/drive/MyDrive/BKQuiz_CPT/MODEL_COMPARISON_4WAYS_20261025')
EXPECTED_ZIP_SHA256='dc4dfe59d228e4490682d41248e00a46ed03b8db9a4e7f59acac6476ac4e2a5b'
ZIP_PATH=DRIVE_ROOT/'bkquiz_four_way_input.zip'
RUN_NAME='four_way_20261025_001'
GENERATION_COUNT=12  # 12 hoặc40; thay RUN_NAME nếu đổi sau khi đã chạy
assert GENERATION_COUNT in (12,40)
assert ZIP_PATH.is_file(), str(ZIP_PATH)

In [ ]:
def file_sha(path):
    with Path(path).open('rb') as f: return hashlib.file_digest(f,'sha256').hexdigest()
LOCAL_ZIP=Path('/content/bkquiz_four_way_input.zip')
shutil.copyfile(ZIP_PATH,LOCAL_ZIP)
assert file_sha(LOCAL_ZIP)==EXPECTED_ZIP_SHA256, 'ZIP/notebook không cùng bản'
WORK=Path('/content/bkquiz_four_way_'+EXPECTED_ZIP_SHA256[:12])
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(LOCAL_ZIP) as z:
    infos=z.infolist(); names=[i.filename for i in infos]
    assert len(names)==len(set(names)) and len(names)<300
    assert sum(i.file_size for i in infos)<512*1024*1024
    for i in infos:
        rel=PurePosixPath(i.filename)
        assert not rel.is_absolute() and '..' not in rel.parts and ':' not in i.filename and chr(92) not in i.filename
        assert stat.S_IFMT(i.external_attr>>16)!=stat.S_IFLNK and i.file_size<128*1024*1024
        assert (WORK/i.filename).resolve().is_relative_to(WORK.resolve())
    m=json.loads(z.read('comparison_manifest.json'))
    assert set(names)=={'comparison_manifest.json',*m['files_sha256']}
    z.extractall(WORK)
for name,wanted in m['files_sha256'].items():
    assert file_sha(WORK/name)==wanted, name
RUNTIME=WORK/'runtime'
RUN=DRIVE_ROOT/'runs'/RUN_NAME
RUN.mkdir(parents=True,exist_ok=True)

## Môi trường
Dùng bootstrap pip đã sửa, giữ CUDA torch của Colab. Log cài đặt được lưu đầy đủ.

In [ ]:
"""Colab environment bootstrap, learned from the reported missing-pip incident.

Standard library only. Importing this file never installs or deletes anything.
"""
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import urllib.request
import uuid

GET_PIP_URL="https://bootstrap.pypa.io/get-pip.py"


def run_logged(command, *, log_path=None, **kwargs):
    """Stream stdout and stderr together; retain the complete error output."""
    if any(key in kwargs for key in ("stdout","stderr","capture_output","text")):
        raise ValueError("run_logged owns output capture")
    kwargs.pop("check",None)
    print("Running:"," ".join(map(str,command)),flush=True)
    process=subprocess.Popen(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1,**kwargs)
    chunks=[]
    with process.stdout:
        for line in process.stdout:
            print(line,end="",flush=True);chunks.append(line)
    code=process.wait();output="".join(chunks)
    if log_path is not None:
        with Path(log_path).open("a",encoding="utf-8") as stream:
            stream.write("COMMAND: "+repr(list(map(str,command)))+"\n"+output+"\nEXIT: "+str(code)+"\n")
    if code:raise subprocess.CalledProcessError(code,command,output=output)
    return subprocess.CompletedProcess(command,code,stdout=output)


def healthy_venv(venv,base_python=sys.executable):
    """An interpreter file alone does not prove that pip or isolation works."""
    venv=Path(venv);python=venv/"bin/python"
    if not python.is_file():return False
    probe="import json,sys;print(json.dumps({'prefix':sys.prefix,'base':sys.base_prefix,'version':list(sys.version_info[:2])}))"
    try:
        own=subprocess.run([str(python),"-c",probe],capture_output=True,text=True,timeout=30)
        base=subprocess.run([str(base_python),"-c",probe],capture_output=True,text=True,timeout=30)
        pip=subprocess.run([str(python),"-m","pip","--version"],capture_output=True,text=True,timeout=30)
        if own.returncode or base.returncode or pip.returncode:return False
        info=json.loads(own.stdout);base_info=json.loads(base.stdout)
        return (Path(info["prefix"]).resolve()==venv.resolve() and info["prefix"]!=info["base"]
                and info["version"]==base_info["version"])
    except (OSError,ValueError,KeyError,subprocess.SubprocessError):return False


def download_get_pip(target):
    """Fetch an official script as a file, not curl|python; record its digest."""
    target=Path(target)
    with urllib.request.urlopen(GET_PIP_URL,timeout=60) as response:
        if not response.geturl().startswith("https://bootstrap.pypa.io/"):
            raise RuntimeError("Unexpected get-pip download destination")
        body=response.read(8*1024*1024+1)
    if not body or len(body)>8*1024*1024:raise RuntimeError("Invalid get-pip download size")
    target.write_bytes(body)
    return hashlib.sha256(body).hexdigest()


def ensure_colab_venv(venv, *, allowed_root=Path("/content"), base_python=sys.executable,
                      allow_system_packages=True, runner=run_logged, downloader=download_get_pip):
    """Repair only this disposable venv; never touch Drive/checkpoints/data."""
    requested=Path(venv);root=Path(allowed_root).resolve();venv=requested.resolve()
    if requested.is_symlink() or venv.parent!=root or venv==root or venv.name in {"drive","datasets","runs","checkpoints"}:
        raise ValueError("Venv must be a dedicated direct child of the allowed scratch directory")
    if healthy_venv(venv,base_python):
        print("Reusing healthy Python + pip:",venv,flush=True)
        return venv/"bin/python"
    if venv.exists():
        if not venv.is_dir() or (any(venv.iterdir()) and not (venv/"pyvenv.cfg").is_file()):
            raise ValueError("Existing path is not a recognizable disposable venv; choose a new venv path")
        # Keep the old environment recoverable. No recursive data deletion.
        backup=venv.with_name(venv.name+".broken."+uuid.uuid4().hex[:12])
        if backup.resolve().parent!=root:raise ValueError("Invalid venv backup target")
        venv.rename(backup);print("Old broken venv retained:",backup,flush=True)
    create=[str(base_python),"-m","venv","--system-site-packages","--without-pip",str(venv)]
    log_path=root/(venv.name+"_setup.log")
    try:runner(create,log_path=log_path)
    except subprocess.CalledProcessError as error:
        details=str(error.output or "")+str(error.stderr or "")
        missing_venv="No module named venv" in details or "No module named 'venv'" in details
        if not missing_venv or not allow_system_packages or not shutil.which("apt-get"):raise
        # Generic python3-venv may target a different interpreter than Colab's.
        version=subprocess.run([str(base_python),"-c","import sys;print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
                               check=True,capture_output=True,text=True).stdout.strip()
        if not __import__('re').fullmatch(r"3\.\d+",version):raise RuntimeError("Cannot identify active Python version")
        runner(["apt-get","update"],log_path=log_path)
        runner(["apt-get","install","-y","python"+version+"-venv"],log_path=log_path)
        runner(create,log_path=log_path)
    python=venv/"bin/python"
    if not python.is_file():raise RuntimeError("Venv creation produced no Python executable; inspect setup log")
    script=root/(venv.name+"_get-pip.py")
    script_sha=downloader(script)
    runner([str(python),str(script)],log_path=log_path)
    if not healthy_venv(venv,base_python):raise RuntimeError("Python/pip health check still failed; stop and inspect setup log")
    (venv/"colab_bootstrap.json").write_text(json.dumps({"base_python":str(base_python),"method":"without-pip_then_official_get-pip",
          "get_pip_url":GET_PIP_URL,"get_pip_sha256":script_sha},indent=2),encoding="utf-8")
    return python

VENV=Path('/content/bkquiz_four_way_venv')
PYTHON=ensure_colab_venv(VENV)
(RUNTIME/'sitecustomize.py').write_text("import importlib.util\n_original=importlib.util.find_spec\ndef _find(name,*args,**kwargs):\n    if name.split('.')[0] in {'torchvision','torchaudio','librosa'}:return None\n    return _original(name,*args,**kwargs)\nimportlib.util.find_spec=_find\n",encoding='utf-8')
ENV=dict(os.environ,PYTHONPATH=str(RUNTIME),TOKENIZERS_PARALLELISM='false')
run_logged([PYTHON,'-m','pip','install','-r',RUNTIME/'requirements.txt'],env=ENV,log_path=RUN/'setup.log')
run_logged([PYTHON,'-c','import torch; assert torch.cuda.is_available(), "Select GPU runtime"; print(torch.__version__,torch.cuda.get_device_properties(0))'],env=ENV,log_path=RUN/'setup.log')
def launch(script,*args):
    run_logged([PYTHON,'-u',RUNTIME/script,*map(str,args)],env=ENV,cwd=RUNTIME,log_path=RUN/'execution.log')
launch('evaluate_four_way.py','--root',WORK,'--mode','preflight')

## Tạo đối chứng SFT-only
Fresh LoRA rank8/alpha16/dropout0,05; không nạp trọng số CPT. Cùng seed42, LR2e-5,
batch1, accumulation8,1epoch, chọn checkpoint theo validation. Khoảng36steps.
Giữ RUN_NAME để resume checkpoint đầy đủ. Nếu chưa tới checkpoint đầu tiên thì cần RUN_NAME mới.

In [ ]:
SFT_ONLY=RUN/'sft_only_training'
args=['--data',WORK/'data','--output',SFT_ONLY,'--mode','train']
if (SFT_ONLY/'run_identity.json').is_file(): args+=['--resume','latest']
launch('train_sft.py',*args)

## Đánh giá bốn bản
Mỗi bản chạy trong một process riêng, cùng NF4, FP16 trên T4, greedy và tắt thinking.
Cache từng câu; không sửa JSON đầu ra rồi chấm lại. Nếu ngắt, Run all với cùng RUN_NAME.
NLL sử dụng ngữ cảnh có đáp án nguồn. MCQ chỉ thấy câu hỏi và phương án, không thấy đáp án/lời giải.

In [ ]:
EVAL=RUN/'evaluation'
for variant in ('base','cpt','sft','cpt_sft'):
    print('\n===== ĐÁNH GIÁ',variant,'=====',flush=True)
    launch('evaluate_four_way.py','--root',WORK,'--output',EVAL,'--sft-run',SFT_ONLY,
           '--mode','evaluate','--variant',variant,'--generation-count',GENERATION_COUNT)
launch('evaluate_four_way.py','--root',WORK,'--output',EVAL,'--sft-run',SFT_ONLY,'--mode','report')

In [ ]:
from IPython.display import Markdown,display
display(Markdown((EVAL/'COMPARISON.md').read_text(encoding='utf-8')))
from google.colab import files
print('Kết quả lưu tại:',EVAL/'four_way_results.zip')
files.download(str(EVAL/'four_way_results.zip'))

## Đọc kết quả
Gửi lại **four_way_results.zip** để phân tích ưu/nhược từng bản và duyệt lời giải.
ZIP gồm bảng số đo, output nguyên văn, identity/hash, gói đánh giá blind và adapter SFT-only mới.
Duyệt blind_review.json trước khi mở blind_review_key.json. Cần xem nội dung trước khi kết luận model nào tốt nhất.
Không tự thay model đang phục vụ. Không phải đo hiệu năng GGUF.